# 🚀 Proyecto Final: Sistema de Gestión de Paquetería Inteligente


## 🧩 Descripción General
Se debe implementar un sistema de gestión de envíos para una empresa de mensajería y logística que permita registrar paquetes, asociarlos a clientes, llevar el control de su estado, y calcular costos según el tipo de envío. El sistema debe incluir relaciones entre clases (composición, agregación, asociación), uso de métodos mágicos, herencia, encapsulación, y polimorfismo.

### 🎯 Objetivo del Proyecto
- Evaluar los siguientes conceptos:

- Clases y objetos

- Atributos públicos, protegidos y privados

- Métodos de instancia, clase y estáticos

- Métodos mágicos (__str__, __init__, etc.)

- Encapsulación con validaciones (getters/setters)

- Asociación, agregación y composición

- Herencia y polimorfismo

- Buenas prácticas (modularidad, documentación)

### ESTRUCTURA E LAS CLASES

| Clase           | Rol                                                                 |
|----------------|----------------------------------------------------------------------|
| Cliente         | Representa a un usuario que envía o recibe paquetes                  |
| Paquete         | Clase base para paquetes, incluye atributos comunes y métodos mágicos |
| PaqueteExpress  | Hereda de Paquete, incluye recargo por urgencia                      |
| PaqueteEstandar | Hereda de Paquete, sin recargo                                       |
| Direccion       | Composición con Cliente (cada cliente tiene una dirección)           |
| Envio           | Agrega un paquete y se asocia a un cliente                           |
| SistemaEnvios   | Administra clientes y envíos                                         |


### 🧪 Requisitos funcionales
1. Registrar clientes y asociarles una dirección.

2. Crear paquetes (express o estándar).

3. Asignar paquetes a envíos y clientes.

4. Calcular el precio de cada envío.

5. Mostrar listado de envíos con información detallada (polimorfismo).

6. Validar datos (precio, peso, nombre del cliente, etc.).

7. Mostrar reporte de todos los envíos realizados.

8. Agregar seguimiento del paquete (pendiente, en tránsito, entregado) usando métodos set_estado().

9. Guardar la información en un archivo .txt o .json.

10. Mostrar totales de ventas por tipo de envío (estándar vs express).

11. Diseñar un menú de opciones para registrar clientes/envíos de forma interactiva.



In [15]:
# Clase que representa la dirección de un cliente
class Direccion:
    def __init__(self, calle, ciudad, codigo_postal):
        self.calle = calle
        self.ciudad = ciudad
        self.codigo_postal = codigo_postal

    def __str__(self):
        return f"{self.calle}, {self.ciudad}, CP: {self.codigo_postal}"

# Clase que representa a un cliente con su dirección
class Cliente:
    def __init__(self, nombre, direccion: Direccion):
        self._nombre = nombre
        self._direccion = direccion

    @property
    def nombre(self):
        return self._nombre

    @property
    def direccion(self):
        return self._direccion

    def __str__(self):
        return f"{self.nombre} - Dirección: {self.direccion}"

# Clase base para todos los paquetes
class Paquete:
    def __init__(self, peso, descripcion):
        self._peso = peso if peso > 0 else 0.1
        self._descripcion = descripcion
        self._estado = "Pendiente"  # Estado inicial

    def set_estado(self, nuevo_estado):
        if nuevo_estado in ["Pendiente", "En tránsito", "Entregado"]:
            self._estado = nuevo_estado

    def get_estado(self):
        return self._estado

    def calcular_precio(self):
        return self._peso * 5000  # Precio base por kilo (en COP)

    def __str__(self):
        return f"{self._descripcion} | {self._peso}kg | Estado: {self._estado}"

# Paquete express con recargo
class PaqueteExpress(Paquete):
    def calcular_precio(self):
        return super().calcular_precio() + 10000  # Recargo fijo en COP

# Paquete estándar sin recargo
class PaqueteEstandar(Paquete):
    def calcular_precio(self):
        return super().calcular_precio()

# Envío que une cliente con un paquete
class Envio:
    def __init__(self, cliente: Cliente, paquete: Paquete):
        self.cliente = cliente
        self.paquete = paquete

    def calcular_total(self):
        return self.paquete.calcular_precio()

    def __str__(self):
        total_cop = f"${self.calcular_total():,.0f}".replace(",", ".")
        return f"Cliente: {self.cliente.nombre}\nPaquete: {self.paquete}\nTotal: {total_cop} COP"

# Sistema que administra clientes y envíos
class SistemaEnvios:
    def __init__(self):
        self.clientes = []
        self.envios = []

    def registrar_cliente(self, nombre, direccion):
        cliente = Cliente(nombre, direccion)
        self.clientes.append(cliente)
        return cliente

    def crear_envio(self, cliente, paquete):
        envio = Envio(cliente, paquete)
        self.envios.append(envio)
        return envio

    def mostrar_envios(self):
        for envio in self.envios:
            print(envio)
            print("-" * 30)

    def resumen_por_tipo(self):
        total_express = sum(e.calcular_total() for e in self.envios if isinstance(e.paquete, PaqueteExpress))
        total_estandar = sum(e.calcular_total() for e in self.envios if isinstance(e.paquete, PaqueteEstandar))
        print(f"Total ventas Express: ${total_express:,.0f} COP".replace(",", "."))
        print(f"Total ventas Estándar: ${total_estandar:,.0f} COP".replace(",", "."))


from datetime import datetime


# Crea un archivo de texto con el resumen de todos los envíos
def generar_reporte_txt(sistema: SistemaEnvios, nombre_archivo="reporte_envios.txt"):
    fecha = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(nombre_archivo, "w", encoding="utf-8") as archivo:
        archivo.write(" SISTEMA DE GESTIÓN DE PAQUETERÍA INTELIGENTE\n")
        archivo.write(f"  Reporte generado: {fecha}\n\n")
        archivo.write("LISTADO DE ENVÍOS\n")
        archivo.write("-" * 50 + "\n")

        for envio in sistema.envios:
            archivo.write(f"Cliente: {envio.cliente.nombre}\n")
            archivo.write(f"Dirección: {envio.cliente.direccion}\n")
            archivo.write(f"Descripción: {envio.paquete._descripcion}\n")
            archivo.write(f"Peso: {envio.paquete._peso} kg\n")
            archivo.write(f"Estado: {envio.paquete.get_estado()}\n")
            tipo = "Express" if isinstance(envio.paquete, PaqueteExpress) else "Estándar"
            archivo.write(f"Tipo de envío: {tipo}\n")
            archivo.write(f"Total: ${envio.calcular_total():,.0f} COP\n".replace(",", "."))
            archivo.write("-" * 50 + "\n")

        total_express = sum(e.calcular_total() for e in sistema.envios if isinstance(e.paquete, PaqueteExpress))
        total_estandar = sum(e.calcular_total() for e in sistema.envios if isinstance(e.paquete, PaqueteEstandar))

        archivo.write("RESUMEN DE VENTAS\n")
        archivo.write(f"-- Total ventas Express: ${total_express:,.0f} COP\n".replace(",", "."))
        archivo.write(f"-- Total ventas Estándar: ${total_estandar:,.0f} COP\n".replace(",", "."))
        archivo.write("-" * 50 + "\n")

    print(f"-- Reporte guardado en: {nombre_archivo}")

# Prueba del sistema completo
sistema = SistemaEnvios()

# Registrar cliente
direccion = Direccion("Trav 58 # 68 H 94", "Bogota", "103")
cliente = sistema.registrar_cliente("Laura Yepes", direccion)

direccion2 = Direccion("Calle 58 f # 47-27", "Bogota", "107")
cliente2 = sistema.registrar_cliente("Manuela Marroquin", direccion)

# Crear paquetes
paquete1 = PaqueteEstandar(2.5, "Bolso ")
paquete2 = PaqueteExpress(1.2, "Maquillaje")

paquete3 = PaqueteEstandar(2.2, "Tenis ")
paquete4 = PaqueteExpress(3.0, "Ropa Femenina")

# Crear envíos
sistema.crear_envio(cliente, paquete1)
sistema.crear_envio(cliente, paquete2)

sistema.crear_envio(cliente2, paquete3)
sistema.crear_envio(cliente2, paquete4)

# Mostrar en consola
sistema.mostrar_envios()

# Guardar en archivo
generar_reporte_txt(sistema)

# Mostrar resumen por tipo
sistema.resumen_por_tipo()

Cliente: Laura Yepes
Paquete: Bolso  | 2.5kg | Estado: Pendiente
Total: $12.500 COP
------------------------------
Cliente: Laura Yepes
Paquete: Maquillaje | 1.2kg | Estado: Pendiente
Total: $16.000 COP
------------------------------
Cliente: Manuela Marroquin
Paquete: Tenis  | 2.2kg | Estado: Pendiente
Total: $11.000 COP
------------------------------
Cliente: Manuela Marroquin
Paquete: Ropa Femenina | 3.0kg | Estado: Pendiente
Total: $25.000 COP
------------------------------
-- Reporte guardado en: reporte_envios.txt
Total ventas Express: $41.000 COP
Total ventas Estándar: $23.500 COP
